##### Callbacks and Observability for monitoring execution

##### First understand the problem.
- suppose we have built a rag chatbot. Architecture looks like ```user-->Retriever-->Prompt-->llm-->Answer``` if user asks ```what is spark streaming``` , bot returns ```I dont know```.
- here the question is ```why did it fail?``` the possiblities reasons are 1.Retriever failed 2.wrong chunks retrieved 3. Prompt issue 4. LLm Issue 5. vecotr DB issue 6. Tool issue. how do we know these right , without observability ```Novisibility```, we dont know what is happend inside the ai system for that we need to have system to track what is happeing inside the chatbot for that reason we need this callbacks.

#### What are Callbacks?
- A callback is function that excecutes automatically when a specific event occurs.
```
Event Happens
      ↓
Callback Executes
```
- When langchain runs, a chain,an llm call, a tool, a retriever- dozsens of things happen internally that we normally cant see. Callbacks are hooks that fire at every meaningful moment in that lifecycle, letting we observe, log, measure, and react to whats happening inside.

```
Without callbacks:          With callbacks:
You send a query            You see:
    ↓                         → Chain started at 14:32:01
[black box]                   → Prompt sent: "You are a..."
    ↓                         → LLM thinking... (token 1, 2, 3...)
You get an answer             → Tool called: search_web("LangChain")
                              → Tool returned in 0.3s
                              → Answer generated: 847ms, 143 tokens
                              → Chain ended: total cost $0.0012
````

- The Key insight: Callbacks Dont Change what your application does - they observe it.

#### The Callback Event Lifecycle

- Every component in langchain fires a predictable set of Events.
```
LLM events:          Chain events:         Tool events:
on_llm_start         on_chain_start        on_tool_start
on_llm_new_token     on_chain_end          on_tool_end
on_llm_end           on_chain_error        on_tool_error
on_llm_error

Retriever events:    Agent events:
on_retriever_start   on_agent_action
on_retriever_end     on_agent_finish
on_retriever_error
```

- Every handler method recieves rich context: the component name,inputs,outputs,timing, and error details.

Level 1 — First Custom Callback Handler


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Any, Dict, List
import time
import os 
from dotenv import load_dotenv
load_dotenv()


class SimpleLoggingHandler(BaseCallbackHandler):
    """The Simplest callback - it will log every event to console."""

    def on_llm_start(self, serialized: Dict, prompts: List[str], **kwargs):
        print(f"🟢 LLM starting...")
        model_name = serialized.get('name', 'unknown') if serialized else 'unknown'
        print(f"   Model: {model_name}")
        print(f"   Prompt preview: {prompts[0][:80]}...")
    
    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="", flush=True)   # stream tokens
    
    def on_llm_end(self, response, **kwargs):
        print(f"\n✅ LLM finished")
        print(f"   Output: {response.generations[0][0].text[:100]}...")
    
    def on_llm_error(self, error: Exception, **kwargs):
        print(f"❌ LLM error: {error}")
    
    def on_chain_start(self, serialized: Dict, inputs: Dict, **kwargs):
        # Safely get the component name from serialized
        chain_name = serialized.get('name', 'unknown') if serialized else 'unknown'
        print(f"\n🔗 Chain started: {chain_name}")
        
        # Safely print input keys or details
        if isinstance(inputs, dict):
            print(f"   Input keys: {list(inputs.keys())}")
        else:
            print(f"   Input: {str(inputs)[:80]}...")
    
    def on_chain_end(self, outputs: Any, **kwargs):
        print(f"🏁 Chain ended")
        
        # Safely handle outputs that might not be dictionaries (e.g. ChatPromptValue, string)
        if isinstance(outputs, dict):
            print(f"   Output keys: {list(outputs.keys())}")
        else:
            print(f"   Output: {str(outputs)[:100]}...")

    def on_chain_error(self, error: Exception, **kwargs):
        print(f"❌ Chain error: {error}")

# Now attach callback to the llm
handler = SimpleLoggingHandler()
llm = ChatGroq(model_name = os.getenv("groq_model_name"),
    temperature = 0,
    callbacks = [handler]
    )

chain = (
    ChatPromptTemplate.from_template("Explain {topic} in 2 Sentences")
    | llm
    | StrOutputParser()
)

result = chain.invoke({"topic":"Langchain Callbacks"},
config = {"callbacks":[handler]}
)



🔗 Chain started: unknown
   Input keys: ['topic']

🔗 Chain started: ChatPromptTemplate
   Input keys: ['topic']
🏁 Chain ended
   Output: messages=[HumanMessage(content='Explain Langchain Callbacks in 2 Sentences', additional_kwargs={}, r...
🟢 LLM starting...
   Model: ChatGroq
   Prompt preview: Human: Explain Langchain Callbacks in 2 Sentences...

✅ LLM finished
   Output: Langchain Callbacks are a feature in the Langchain framework that allows developers to execute custo...

🔗 Chain started: unknown
   Input: content="Langchain Callbacks are a feature in the Langchain framework that allow...
🏁 Chain ended
   Output: Langchain Callbacks are a feature in the Langchain framework that allows developers to execute custo...
🏁 Chain ended
   Output: Langchain Callbacks are a feature in the Langchain framework that allows developers to execute custo...


Level 2 — Timing and Cost Tracking Handler

In [9]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time
from typing import Any, Dict, List
from uuid import UUID

# Token cost table (per 1000 tokens)
TOKEN_COSTS = {
    "gpt-4o":        {"input": 0.005,  "output": 0.015},
    "gpt-4o-mini":   {"input": 0.00015,"output": 0.0006},
    "gpt-4-turbo":   {"input": 0.01,   "output": 0.03},
    "claude-3-haiku":{"input": 0.00025,"output": 0.00125},
}

class CostAndTimingHandler(BaseCallbackHandler):
    """Track token usage, cost, and timing for every LLM call."""

    def __init__(self):
        self.calls = []
        self._current = {}

    def on_llm_start(self, serialized, prompts, **kwargs):
        run_id = str(kwargs.get("run_id", ""))
        self._current[run_id] = {
            "start": time.time(),
            "model": serialized.get("kwargs", {}).get("model_name", "unknown"),
            "prompt_preview": prompts[0][:100] if prompts else ""
        }

    def on_llm_end(self, response, **kwargs):
        run_id = str(kwargs.get("run_id", ""))
        call = self._current.pop(run_id, {})
        elapsed = time.time() - call.get("start", time.time())

        # Extract token usage from response
        usage = {}
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage", {})

        prompt_tokens     = usage.get("prompt_tokens", 0)
        completion_tokens = usage.get("completion_tokens", 0)
        total_tokens      = usage.get("total_tokens", 0)

        # Calculate cost
        model = call.get("model", "gpt-4o-mini")
        costs = TOKEN_COSTS.get(model, TOKEN_COSTS["gpt-4o-mini"])
        input_cost  = (prompt_tokens / 1000) * costs["input"]
        output_cost = (completion_tokens / 1000) * costs["output"]
        total_cost  = input_cost + output_cost

        record = {
            "model":              model,
            "elapsed_s":          round(elapsed, 3),
            "prompt_tokens":      prompt_tokens,
            "completion_tokens":  completion_tokens,
            "total_tokens":       total_tokens,
            "cost_usd":           round(total_cost, 6),
        }
        self.calls.append(record)

        print(f"\n📊 LLM call complete:")
        print(f"   Model:    {model}")
        print(f"   Time:     {elapsed:.2f}s")
        print(f"   Tokens:   {prompt_tokens} in + {completion_tokens} out = {total_tokens} total")
        print(f"   Cost:     ${total_cost:.6f}")

    def summary(self):
        if not self.calls:
            print("No calls recorded.")
            return
        total_cost   = sum(c["cost_usd"] for c in self.calls)
        total_tokens = sum(c["total_tokens"] for c in self.calls)
        total_time   = sum(c["elapsed_s"] for c in self.calls)
        print(f"\n{'='*40}")
        print(f"📈 Session Summary ({len(self.calls)} LLM calls)")
        print(f"   Total tokens:  {total_tokens:,}")
        print(f"   Total cost:    ${total_cost:.6f}")
        print(f"   Total time:    {total_time:.2f}s")
        print(f"   Avg per call:  ${total_cost/len(self.calls):.6f}")
        print(f"{'='*40}")

# Usage
tracker = CostAndTimingHandler()
chain = ChatPromptTemplate.from_template("Explain {topic}") | llm | StrOutputParser()

# Pass the tracker to each invoke call via config
chain.invoke({"topic": "vector embeddings"}, config={"callbacks": [tracker]})
chain.invoke({"topic": "RAG pipelines"}, config={"callbacks": [tracker]})
chain.invoke({"topic": "LangChain agents"}, config={"callbacks": [tracker]})

tracker.summary()
# ========================================
# 📈 Session Summary (3 LLM calls)
#    Total tokens:  847
#    Total cost:    $0.000423
#    Total time:    4.21s
#    Avg per call:  $0.000141
# ========================================

🟢 LLM starting...
   Model: ChatGroq
   Prompt preview: Human: Explain vector embeddings...

📊 LLM call complete:
   Model:    llama-3.3-70b-versatile
   Time:     2.74s
   Tokens:   39 in + 789 out = 828 total
   Cost:     $0.000479

✅ LLM finished
   Output: Vector embeddings, also known as vector representations or dense embeddings, are a way to represent ...
🟢 LLM starting...
   Model: ChatGroq
   Prompt preview: Human: Explain RAG pipelines...

📊 LLM call complete:
   Model:    llama-3.3-70b-versatile
   Time:     2.48s
   Tokens:   40 in + 685 out = 725 total
   Cost:     $0.000417

✅ LLM finished
   Output: RAG (Retrieve, Augment, Generate) pipelines are a type of natural language processing (NLP) architec...
🟢 LLM starting...
   Model: ChatGroq
   Prompt preview: Human: Explain LangChain agents...

📊 LLM call complete:
   Model:    llama-3.3-70b-versatile
   Time:     1.77s
   Tokens:   40 in + 660 out = 700 total
   Cost:     $0.000402

✅ LLM finished
   Output: LangChain agen

Level 3 — Full Observability Handler (Real World)

In [10]:
from langchain_core.callbacks import BaseCallbackHandler
import time, json, logging
from datetime import datetime
from typing import Any, Dict, List, Optional
from uuid import UUID

# Set up structured logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler("langchain_audit.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("langchain.observability")

class FullObservabilityHandler(BaseCallbackHandler):
    """
    Production-grade observability handler.
    Tracks: timing, tokens, costs, errors, tool calls, chain hierarchy.
    Logs to file + console. Stores all events for analysis.
    """

    def __init__(self, session_id: str = "default", user_id: str = "anonymous"):
        self.session_id  = session_id
        self.user_id     = user_id
        self.events      = []
        self._timers     = {}
        self.total_cost  = 0.0
        self.total_tokens = 0

    def _log(self, event_type: str, data: Dict, run_id: str = ""):
        event = {
            "timestamp":  datetime.now().isoformat(),
            "session_id": self.session_id,
            "user_id":    self.user_id,
            "run_id":     str(run_id),
            "event_type": event_type,
            **data
        }
        self.events.append(event)
        logger.info(json.dumps(event))
        return event

    # ── LLM Events ────────────────────────────────────────
    def on_llm_start(self, serialized, prompts, run_id=None, **kwargs):
        self._timers[str(run_id)] = time.time()
        self._log("llm_start", {
            "model":          serialized.get("kwargs", {}).get("model_name", "unknown"),
            "num_prompts":    len(prompts),
            "prompt_length":  len(prompts[0]) if prompts else 0
        }, run_id)

    def on_llm_end(self, response, run_id=None, **kwargs):
        elapsed = time.time() - self._timers.pop(str(run_id), time.time())
        usage   = {}
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage", {})

        tokens = usage.get("total_tokens", 0)
        cost   = (tokens / 1000) * 0.0006  # simplified cost
        self.total_tokens += tokens
        self.total_cost   += cost

        self._log("llm_end", {
            "elapsed_s":          round(elapsed, 3),
            "total_tokens":       tokens,
            "prompt_tokens":      usage.get("prompt_tokens", 0),
            "completion_tokens":  usage.get("completion_tokens", 0),
            "cost_usd":           round(cost, 6),
        }, run_id)

    def on_llm_error(self, error, run_id=None, **kwargs):
        elapsed = time.time() - self._timers.pop(str(run_id), time.time())
        self._log("llm_error", {
            "error_type":    type(error).__name__,
            "error_message": str(error),
            "elapsed_s":     round(elapsed, 3)
        }, run_id)
        logger.error(f"LLM ERROR in session {self.session_id}: {error}")

    # ── Chain Events ───────────────────────────────────────
    def on_chain_start(self, serialized, inputs, run_id=None, **kwargs):
        self._timers[str(run_id)] = time.time()
        self._log("chain_start", {
            "chain_name": serialized.get("name", "unknown"),
            "input_keys": list(inputs.keys()) if isinstance(inputs, dict) else []
        }, run_id)

    def on_chain_end(self, outputs, run_id=None, **kwargs):
        elapsed = time.time() - self._timers.pop(str(run_id), time.time())
        self._log("chain_end", {
            "elapsed_s":   round(elapsed, 3),
            "output_keys": list(outputs.keys()) if isinstance(outputs, dict) else []
        }, run_id)

    def on_chain_error(self, error, run_id=None, **kwargs):
        self._log("chain_error", {
            "error_type":    type(error).__name__,
            "error_message": str(error)
        }, run_id)

    # ── Tool Events ────────────────────────────────────────
    def on_tool_start(self, serialized, input_str, run_id=None, **kwargs):
        self._timers[str(run_id)] = time.time()
        self._log("tool_start", {
            "tool_name":   serialized.get("name", "unknown"),
            "tool_input":  str(input_str)[:200]
        }, run_id)

    def on_tool_end(self, output, run_id=None, **kwargs):
        elapsed = time.time() - self._timers.pop(str(run_id), time.time())
        self._log("tool_end", {
            "elapsed_s":   round(elapsed, 3),
            "output_length": len(str(output)),
            "output_preview": str(output)[:200]
        }, run_id)

    def on_tool_error(self, error, run_id=None, **kwargs):
        self._log("tool_error", {
            "error_type":    type(error).__name__,
            "error_message": str(error)
        }, run_id)

    # ── Retriever Events ───────────────────────────────────
    def on_retriever_start(self, serialized, query, run_id=None, **kwargs):
        self._timers[str(run_id)] = time.time()
        self._log("retriever_start", {
            "query":          query,
            "query_length":   len(query)
        }, run_id)

    def on_retriever_end(self, documents, run_id=None, **kwargs):
        elapsed = time.time() - self._timers.pop(str(run_id), time.time())
        self._log("retriever_end", {
            "elapsed_s":      round(elapsed, 3),
            "docs_returned":  len(documents),
            "total_chars":    sum(len(d.page_content) for d in documents)
        }, run_id)

    # ── Agent Events ───────────────────────────────────────
    def on_agent_action(self, action, run_id=None, **kwargs):
        self._log("agent_action", {
            "tool":      action.tool,
            "tool_input": str(action.tool_input)[:200]
        }, run_id)

    def on_agent_finish(self, finish, run_id=None, **kwargs):
        self._log("agent_finish", {
            "output_preview": str(finish.return_values)[:200]
        }, run_id)

    # ── Analysis ───────────────────────────────────────────
    def get_session_report(self) -> Dict:
        by_type = {}
        for e in self.events:
            t = e["event_type"]
            by_type[t] = by_type.get(t, 0) + 1

        errors = [e for e in self.events if "error" in e["event_type"]]
        tool_events = [e for e in self.events if e["event_type"] == "tool_start"]

        return {
            "session_id":   self.session_id,
            "total_events": len(self.events),
            "events_by_type": by_type,
            "total_tokens": self.total_tokens,
            "total_cost_usd": round(self.total_cost, 6),
            "error_count":  len(errors),
            "tool_calls":   len(tool_events),
            "tools_used":   list({e["tool_name"] for e in tool_events if "tool_name" in e})
        }

Level 4 — Where to Attach Callbacks (3 levels)


In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

handler = FullObservabilityHandler(session_id="demo", user_id="subbu")

# ── Level 1: On the LLM (fires only for LLM events) ──────
llm_with_cb = ChatGroq(
    model_name=os.getenv("groq_model_name"),
    callbacks=[handler]  # ← only LLM events
)

# ── Level 2: On invoke (fires for this one call only) ─────
chain = (
    ChatPromptTemplate.from_template("Explain {topic}")
    |  ChatGroq(model_name=os.getenv("groq_model_name"))
    | StrOutputParser()
)

result = chain.invoke(
    {"topic": "embeddings"},
    config={"callbacks": [handler]}   # ← only this invoke
)

# ── Level 3: On the chain itself (fires for all invokes) ──
chain_with_cb = (
    ChatPromptTemplate.from_template("Explain {topic}")
    | ChatGroq(model_name=os.getenv("groq_model_name"))
    | StrOutputParser()
).with_config({"callbacks": [handler]})  # ← always fires

result = chain_with_cb.invoke({"topic": "RAG"})

# ── Level 4: Global — via RunnableConfig ──────────────────
from langchain_core.runnables import RunnableConfig

config = RunnableConfig(callbacks=[handler])
result = chain.invoke({"topic": "agents"}, config=config)

2026-06-25 11:45:45,307 | WARNING | Error in FullObservabilityHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
2026-06-25 11:45:45,315 | INFO | {"timestamp": "2026-06-25T11:45:45.313838", "session_id": "demo", "user_id": "subbu", "run_id": "019efd6b-8fa3-78d2-be0c-76f23a6a1321", "event_type": "chain_start", "chain_name": "ChatPromptTemplate", "input_keys": ["topic"]}
2026-06-25 11:45:45,315 | INFO | {"timestamp": "2026-06-25T11:45:45.315590", "session_id": "demo", "user_id": "subbu", "run_id": "019efd6b-8fa3-78d2-be0c-76f23a6a1321", "event_type": "chain_end", "elapsed_s": 0.002, "output_keys": []}
2026-06-25 11:45:45,324 | INFO | {"timestamp": "2026-06-25T11:45:45.324079", "session_id": "demo", "user_id": "subbu", "run_id": "019efd6b-8faa-7991-b1c0-fa53fe59dce4", "event_type": "llm_start", "model": "llama-3.3-70b-versatile", "num_prompts": 1, "prompt_length": 25}


2026-06-25 11:45:47,428 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-25 11:45:47,428 | INFO | {"timestamp": "2026-06-25T11:45:47.428514", "session_id": "demo", "user_id": "subbu", "run_id": "019efd6b-8faa-7991-b1c0-fa53fe59dce4", "event_type": "llm_end", "elapsed_s": 2.113, "total_tokens": 714, "prompt_tokens": 38, "completion_tokens": 676, "cost_usd": 0.000428}
2026-06-25 11:45:47,428 | WARNING | Error in FullObservabilityHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
2026-06-25 11:45:47,442 | INFO | {"timestamp": "2026-06-25T11:45:47.440918", "session_id": "demo", "user_id": "subbu", "run_id": "019efd6b-97ef-7fb0-97e7-36f176190a5b", "event_type": "chain_end", "elapsed_s": 0.012, "output_keys": []}
2026-06-25 11:45:47,444 | INFO | {"timestamp": "2026-06-25T11:45:47.444502", "session_id": "demo", "user_id": "subbu", "run_id": "019efd6b-8f9c-7bd0-995a-21a51a2371ca", "event_type": "chain

Level 5 — LangSmith Integration (the gold standard for observability)
- LangSmith is Anthropic's hosted observability platform for LangChain. Every run is automatically traced with full input/output, timing, token costs, and error details — with zero code changes.


In [ ]:
import os

# Set these environment variables to enable LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"]    = "your-langsmith-api-key"
os.environ["LANGCHAIN_PROJECT"]    = "my-rag-application"
os.environ["LANGCHAIN_ENDPOINT"]   = "https://api.smith.langchain.com"

# That's it. Every chain/agent/LLM call is now automatically traced.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("Answer this: {question}")
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

# This call is automatically logged to LangSmith — no callbacks needed
result = chain.invoke({"question": "What is RAG?"})

# In LangSmith you'll see:
# ✅ Full prompt sent to the LLM
# ✅ Full response received
# ✅ Latency breakdown per component
# ✅ Token counts and costs
# ✅ Chain hierarchy (which step called which)
# ✅ Any errors with full stack traces

Adding custom metadata to traces:


In [ ]:
from langchain_core.runnables import RunnableConfig
from langsmith import Client

# Add metadata to any run — shows up in LangSmith dashboard
config = RunnableConfig(
    run_name="customer_query",          # label for this run
    tags=["production", "v2.1"],        # filter in dashboard
    metadata={
        "user_id":    "subbu_001",
        "session_id": "sess_abc123",
        "query_type": "technical",
        "ab_variant": "prompt_v2"
    }
)

result = chain.invoke({"question": "Explain LCEL"}, config=config)

# Now in LangSmith you can:
# - Filter all runs by user_id="subbu_001"
# - Compare latency between ab_variant="prompt_v1" vs "prompt_v2"
# - See error rate by query_type

LangSmith evaluation:


In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()

# Define an evaluator — grades your chain's answers
def correctness_evaluator(run, example):
    """Checks if the chain's answer matches the expected answer."""
    chain_answer = run.outputs.get("output", "")
    expected     = example.outputs.get("answer", "")
    score = 1.0 if expected.lower() in chain_answer.lower() else 0.0
    return {"key": "correctness", "score": score}

# Run evaluation against a test dataset
results = evaluate(
    lambda inputs: chain.invoke(inputs),
    data="my-test-dataset",             # dataset created in LangSmith UI
    evaluators=[correctness_evaluator],
    experiment_prefix="rag-v2-test"
)

print(results.to_pandas())

#### What is Observability?
- Observability means being able to understand what happened inside your AI system by examining its outputs, traces, logs, and metrics.
- think like ```Callbacks --> Logs --> Observability```. so here Callbacks Generates data. Observability helps analyze it.
- There are three pillars of Observability: 
1. logs : Text records eg: LLm started,Prompt Sent, Response Generated.
2. Metrics : Numbers eg: latency, token count, cost, error rate.
3. Traces : Complete Excecution path. user-->retriever-->prompt-->llm-->tool-->Answer. every step will be visible.

#### Overview
Callbacks in LangChain are event-driven handlers that execute when specific actions occur during chain, model, retriever, tool, or agent execution. They are used for logging, monitoring, token tracking, cost analysis, and debugging. Observability refers to the broader capability of understanding system behavior through logs, metrics, and traces. LangSmith is LangChain's observability platform that provides end-to-end tracing, monitoring, prompt inspection, and evaluation for RAG and agent-based applications.